In [ ]:
import pandas as pd
import yfinance as yf
from google.cloud import bigquery

PROJECT_ID = "fluid-terminal-465516-s7"
DATASET = "magic_formula"
TABLE = "benchmark_daily_price"

START = "2006-01-01"
END   = "2025-12-31"

# Download benchmark
df = yf.download(
    "^SP500TR",
    start=START,
    end=END,
    auto_adjust=False,
    progress=False
)

if df.empty:
    raise RuntimeError("No data downloaded")

# 🔑 FIX: flatten MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]

df = df.reset_index()
df.rename(columns={"Date": "date"}, inplace=True)

df = df[["date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]].copy()
df.rename(columns={
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Adj Close": "adj_close",
    "Volume": "volume"
}, inplace=True)

df["symbol"] = "SP500TR"

# Enforce types
df["date"] = pd.to_datetime(df["date"]).dt.date
for c in ["open", "high", "low", "close", "adj_close"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["volume"] = pd.to_numeric(df["volume"], errors="coerce")

df = df.dropna(subset=["adj_close"])

print(df.head())
print(df.tail())
print("Rows:", len(df))

# Upload to BigQuery
client = bigquery.Client(project=PROJECT_ID)
table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"

job = client.load_table_from_dataframe(
    df,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
)
job.result()

print("Upload complete:", table_id)